# Blood Bank Master Dataset — Data Cleaning Pipeline

**Source file:** `blood-bank-master.csv`  
**Records:** 2,823 rows × 27 columns  
**Scope:** 35 States/UTs across India  
**Cleaned by:** Automated Python pipeline  
**Date:** June 2026

---

## Fixes applied in this notebook

| # | Fix | Method |
|---|-----|--------|
| 1 | Apheresis invalid value `"2"` → `"UNKNOWN"` | Replace |
| 2 | Corrupted phone fields (lakh-comma formatting) | Strip commas, validate |
| 3 | Pincode — stray internal spaces | Remove spaces |
| 4 | Pincode — invalid/non-6-digit values | Flag as `INVALID` |
| 5 | City name — case-only variants | Normalise to Title Case |
| 6 | City name — confirmed spelling variants (same city/state) | Canonical mapping |
| 7 | District casing — mixed ALL CAPS / Title Case | Normalise to Title Case |
| 8 | HTML entity artifacts (`&#39;` → `'`) | Unescape across all text fields |
| 9 | Broken emails (missing `@`, merged, wrong separator, etc.) | Manual fix map + flag |
| 10 | Service Time — 24×7 spelling variants | Normalise to `24x7` |
| 11 | Missing-value tokens (`NA`, `N/A`, `na`, `N-A`) | Replace with blank (NaN) |
| 12 | Date formats — normalise to `DD-MM-YYYY` + extract renewal status | Parse & reformat |
| 13 | Self-flagged `(Repeated)` / `(REPEATED)` rows | Remove duplicates |
| 14 | Global casing uniformity across all text columns | Title Case / UPPER per column logic |

---

## 0. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import re
import html
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

INPUT_FILE  = 'blood-bank-master.csv'
OUTPUT_FILE = 'blood_bank_master_clean.csv'

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 1. Load Data

In [2]:
df = pd.read_csv(INPUT_FILE, encoding='utf-8', low_memory=False, dtype=str)

# Strip leading/trailing whitespace from all column names
df.columns = df.columns.str.strip()

# Strip leading/trailing whitespace from all cell values
df = df.apply(lambda col: col.str.strip() if col.dtype == object else col)

print(f'Loaded: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

Loaded: 2823 rows x 27 columns
Columns: ['Sr No', 'Blood Bank Name', 'State', 'District', 'City', 'Address', 'Pincode', 'Contact No', 'Mobile', 'Helpline', 'Fax', 'Email', 'Website', 'Nodal Officer', 'Contact Nodal Officer', 'Mobile Nodal Officer', 'Email Nodal Officer', 'Qualification Nodal Officer', 'Category', 'Blood Component Available', 'Apheresis', 'Service Time', 'License #', 'Date License Obtained', 'Date of Renewal', 'Latitude', 'Longitude']


---
## Fix 1 — Apheresis: Invalid Value `"2"` → `"UNKNOWN"`

The `Apheresis` field should contain only `YES` or `NO`.  
29 rows contain `"2"` — likely a numeric encoding artefact from the UP state batch import.  
These cannot be safely resolved to YES/NO without the original source, so they are set to `UNKNOWN`.

In [3]:
before = (df['Apheresis'] == '2').sum()

df['Apheresis'] = df['Apheresis'].replace('2', 'UNKNOWN')

after = (df['Apheresis'] == '2').sum()
print(f'Apheresis "2" values: {before} → {after} remaining')
print(df['Apheresis'].value_counts(dropna=False))

Apheresis "2" values: 29 → 0 remaining
Apheresis
NO         1612
NaN         768
YES         414
UNKNOWN      29
Name: count, dtype: int64


---
## Fix 2 — Corrupted Phone Fields (Lakh-Comma Formatting Artefact)

14 rows have phone-type fields mangled into lakh-style comma groups  
(e.g. `91,77,23,43,07,09,39,30,00,000`).  
Fix: strip all commas; if the resulting digit string is a valid 10-digit mobile or known format, keep it — otherwise flag as `INVALID_PHONE`.

In [4]:
PHONE_COLS = ['Contact No', 'Mobile', 'Helpline', 'Fax',
              'Contact Nodal Officer', 'Mobile Nodal Officer']

def fix_phone(val):
    """Remove lakh-style commas from a phone number string."""
    if pd.isna(val) or str(val).strip().upper() in ('NA', 'N/A', 'N-A', 'NAN', ''):
        return val
    v = str(val).strip()
    # Detect lakh-comma pattern: groups of 1-2 digits separated by commas ending in zeros
    if re.match(r'^[\d,]+$', v) and v.count(',') >= 4:
        stripped = v.replace(',', '')
        # Trim leading country code artifacts (e.g. 91 prefix on Indian numbers)
        if len(stripped) == 12 and stripped.startswith('91'):
            stripped = stripped[2:]
        if len(stripped) == 10 and stripped.isdigit():
            return stripped
        return 'INVALID_PHONE'
    return v

fixed_count = 0
for col in PHONE_COLS:
    if col not in df.columns:
        continue
    mask = df[col].apply(
        lambda v: bool(pd.notna(v) and re.match(r'^[\d,]+$', str(v).strip()) and str(v).count(',') >= 4)
    )
    fixed_count += mask.sum()
    df.loc[mask, col] = df.loc[mask, col].apply(fix_phone)

print(f'Phone fields repaired: {fixed_count} cells across {PHONE_COLS}')

Phone fields repaired: 16 cells across ['Contact No', 'Mobile', 'Helpline', 'Fax', 'Contact Nodal Officer', 'Mobile Nodal Officer']


---
## Fix 3 — Pincode: Remove Stray Internal Spaces

16 pincodes contain an embedded space (e.g. `520 008`).  
These are cosmetically incorrect but the digits are valid — spaces are simply removed.

In [5]:
before_spaces = df['Pincode'].str.contains(r'\s', na=False).sum()

df['Pincode'] = df['Pincode'].str.replace(r'\s+', '', regex=True)

after_spaces = df['Pincode'].str.contains(r'\s', na=False).sum()
print(f'Pincodes with stray spaces: {before_spaces} → {after_spaces}')

Pincodes with stray spaces: 16 → 0


---
## Fix 4 — Pincode: Flag Invalid / Non-6-Digit Values

6 pincodes are genuinely invalid after space removal:  
single/double digit values (`1`, `16`, `20`, `34`) likely truncated during entry,  
and two 7-digit values (`8441010`, `2700274`).  
These cannot be auto-corrected — flagged as `INVALID_PINCODE`.

In [6]:
def validate_pincode(v):
    if pd.isna(v) or str(v).strip() == '':
        return v
    v = str(v).strip().rstrip(',')
    if re.match(r'^\d{6}$', v):
        return v
    return 'INVALID_PINCODE'

before_invalid = df['Pincode'].apply(
    lambda v: pd.notna(v) and str(v).strip() not in ('', 'nan') and not re.match(r'^\d{6}$', str(v).strip())
).sum()

df['Pincode'] = df['Pincode'].apply(validate_pincode)

flagged = (df['Pincode'] == 'INVALID_PINCODE').sum()
print(f'Invalid pincodes before: {before_invalid} | Flagged as INVALID_PINCODE: {flagged}')

Invalid pincodes before: 7 | Flagged as INVALID_PINCODE: 6


---
## Fix 5 — City: Case-Only Variants → Title Case

Several cities appear in both ALL CAPS and Title Case forms within the same state  
(e.g. `KOLKATA` / `Kolkata`, `HYDERABAD` / `Hyderabad`, `ThiruvAnanthapuram` / `Thiruvananthapuram`).  
All City values are normalised to Title Case as a base step before spelling fixes.

In [7]:
before_unique = df['City'].nunique()

df['City'] = df['City'].str.strip().str.title()

after_unique = df['City'].nunique()
print(f'City unique values: {before_unique} → {after_unique} (after Title Case normalisation)')

City unique values: 1121 → 1109 (after Title Case normalisation)


---
## Fix 6 — City: Confirmed Spelling Variants → Canonical Form

30 groups confirmed to be the same city in the same state with different spellings.  
Each is mapped to a single canonical name (verified against official government naming).  
Note: `Ananthapur → Anantapur` (Andhra Pradesh only); `Anandapur` (Odisha) is a different city and is NOT changed.

In [8]:
# Mapping: variant → canonical
# Applied ONLY within the correct state to avoid cross-state collisions
CITY_STATE_FIX = [
    # (state,              variant,             canonical)
    ('Andhra Pradesh',  'Ananthapur',          'Anantapur'),
    ('Karnataka',       'Bhadravathi',         'Bhadravati'),
    ('Uttar Pradesh',   'Bijnore',             'Bijnor'),
    ('Maharashtra',     'Buldana',             'Buldhana'),
    ('Kerala',          'Chegannur',           'Chengannur'),
    ('Uttar Pradesh',   'Frukhabad',           'Farrukhabad'),
    ('Uttar Pradesh',   'Gautam Budh Nagar',   'Gautam Buddha Nagar'),
    ('Gujarat',         'Godhara',             'Godhra'),
    ('Telangana',       'Himayathnagar',       'Himayatnagar'),
    ('Telangana',       'Hymayathnagar',       'Himayatnagar'),
    ('Tamil Nadu',      'Kanya Kumari',        'Kanyakumari'),
    ('Kerala',          'Kottiyam',            'Kottayam'),
    ('Andhra Pradesh',  'Madnapalli',          'Madanapalli'),
    ('Tamil Nadu',      'Madurantakam',        'Maduranthagam'),
    ('Telangana',       'Mahaboobnagar',       'Mahabubnagar'),
    ('Tamil Nadu',      'Nagarcoil',           'Nagercoil'),
    ('Tamil Nadu',      'Nagerkoil',           'Nagercoil'),
    ('Kerala',          'Pala',                'Palai'),
    ('Tamil Nadu',      'Pattukkottai',        'Pattukottai'),
    ('Kerala',          'Perintalmanna',       'Perinthalmanna'),
    ('Kerala',          'Perinthanmanna',      'Perinthalmanna'),
    ('Uttar Pradesh',   'Shahjhanpur',         'Shahjahanpur'),
    ('Karnataka',       'Shimoga',             'Shivamogga'),
    ('Odisha',          'Sundargarh',          'Sundergarh'),
    ('Kerala',          'Thalasherry',         'Thalassery'),
    ('Kerala',          'Thellakam',           'Thellakom'),
    ('Tamil Nadu',      'Thiruchengode',       'Tiruchengkodu'),
    ('Tamil Nadu',      'Thirunelveli',        'Tirunelveli'),
    ('Tamil Nadu',      'Thiruppur',           'Tiruppur'),
    ('Tamil Nadu',      'Tirchy',              'Trichy'),
    ('Uttar Pradesh',   'Bhadoi',              'Bhadohi')
]

fix_count = 0
for state, variant, canonical in CITY_STATE_FIX:
    # After title-casing, match accordingly
    variant_title   = variant.title()
    canonical_title = canonical.title()
    state_title     = state.title()
    mask = (df['City'] == variant_title) & (df['State'].str.title() == state_title)
    n = mask.sum()
    if n > 0:
        df.loc[mask, 'City'] = canonical_title
        fix_count += n
        print(f'  [{state}] {variant_title!r} → {canonical_title!r}  ({n} rows)')

print(f'\nTotal city rows corrected: {fix_count}')

  [Andhra Pradesh] 'Ananthapur' → 'Anantapur'  (2 rows)
  [Karnataka] 'Bhadravathi' → 'Bhadravati'  (1 rows)
  [Uttar Pradesh] 'Bijnore' → 'Bijnor'  (2 rows)
  [Maharashtra] 'Buldana' → 'Buldhana'  (1 rows)
  [Kerala] 'Chegannur' → 'Chengannur'  (1 rows)
  [Uttar Pradesh] 'Frukhabad' → 'Farrukhabad'  (1 rows)
  [Uttar Pradesh] 'Gautam Budh Nagar' → 'Gautam Buddha Nagar'  (2 rows)
  [Gujarat] 'Godhara' → 'Godhra'  (1 rows)
  [Telangana] 'Himayathnagar' → 'Himayatnagar'  (1 rows)
  [Telangana] 'Hymayathnagar' → 'Himayatnagar'  (1 rows)
  [Tamil Nadu] 'Kanya Kumari' → 'Kanyakumari'  (1 rows)
  [Kerala] 'Kottiyam' → 'Kottayam'  (1 rows)
  [Andhra Pradesh] 'Madnapalli' → 'Madanapalli'  (1 rows)
  [Tamil Nadu] 'Madurantakam' → 'Maduranthagam'  (1 rows)
  [Telangana] 'Mahaboobnagar' → 'Mahabubnagar'  (1 rows)
  [Tamil Nadu] 'Nagarcoil' → 'Nagercoil'  (1 rows)
  [Tamil Nadu] 'Nagerkoil' → 'Nagercoil'  (1 rows)
  [Kerala] 'Pala' → 'Palai'  (1 rows)
  [Tamil Nadu] 'Pattukkottai' → 'Pattukottai' 

---
## Fix 7 — District: Normalise Casing → Title Case

2,668 of 2,822 non-null District values are ALL CAPS; 154 are Title/Mixed case.  
All are normalised to Title Case for readability and consistency.

In [9]:
before_caps = (df['District'] == df['District'].str.upper()).sum()

df['District'] = df['District'].str.strip().str.title()

print(f'Districts previously ALL CAPS: {before_caps}')
print(f'Sample after Title Case normalisation:')
print(df['District'].dropna().unique()[:10].tolist())

Districts previously ALL CAPS: 2668
Sample after Title Case normalisation:
['South Andaman', 'Anantapur', 'Chittoor', 'East Godavari', 'Guntur', 'Krishna', 'Kurnool', 'Prakasam', 'Spsr Nellore', 'Srikakulam']


---
## Fix 8 — HTML Entity Unescaping (`&#39;` → `'`)

46 rows (50 field instances) contain literal `&#39;` instead of an apostrophe.  
Applied across: Blood Bank Name, Address, Email, Nodal Officer, Service Time.

In [10]:
HTML_COLS = ['Blood Bank Name', 'Address', 'Email',
             'Nodal Officer', 'Service Time']

total_fixed = 0
for col in HTML_COLS:
    if col not in df.columns:
        continue
    mask = df[col].str.contains('&#', na=False)
    count = mask.sum()
    if count > 0:
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda v: html.unescape(str(v)) if pd.notna(v) else v
        )
        total_fixed += count
        print(f'  {col}: {count} rows unescaped')

print(f'\nTotal HTML entity rows fixed: {total_fixed}')

  Blood Bank Name: 21 rows unescaped
  Address: 25 rows unescaped
  Email: 1 rows unescaped
  Nodal Officer: 1 rows unescaped
  Service Time: 2 rows unescaped

Total HTML entity rows fixed: 50


---
## Fix 9 — Emails: Manual Fix Map for Structurally Broken Addresses

16 emails are structurally broken (missing `@`, merged double-addresses, wrong separator,  
comma-as-dot, or not an email at all like `"yes"`).  

- Fixable ones are corrected directly.  
- Unresolvable ones (no `@` with no recovery path) are set to `EMAIL_INVALID`.  

Note: The remaining ~108 rows with multiple valid emails separated by `, ` are **not** broken —  
they are valid multi-address entries and are left as-is.

In [11]:
# Manual fix map: Sr No → corrected value (None = EMAIL_INVALID)
EMAIL_MANUAL_FIX = {
    13:   None,                                                                          # Plain text, no email
    20:   None,                                                                          # 'svimshosp' — no @ and unresolvable
    61:   'vijayasribloodbankvja@gmail.com',                                             # Space before @gmail removed
    76:   'knlgghbloodbank@gmail.com, supdtgghknl@gmail.com',                           # Merged — split at .com boundary
    152:  None,                                                                          # 'dmoykggmail.com' — missing @, unresolvable
    353:  'bloodbankrjn@gmail.com',                                                      # 'blood bankrjn@gmail.com' — space in local part
    465:  'adgbloodbank@charutarhealth.org, sadhanas@charutarhealth.org',               # '/' separator → ','
    467:  'glh_deesa@yahoo.com, drjagdish2000@yahoo.co.in',                             # space-only separator → ', '
    702:  None,                                                                          # 'yes' — not an email
    1346: 'iimsk11@gmail.com',                                                           # Second address truncated ('iimsr11@gmail.') — removed
    1361: 'bhalbloodbank@rediffmail.com',                                                # First address truncated ('ircslatur@gmail') — removed
    1952: 'bloodbank@mmm.org.in',                                                        # 'bloodbank @mmm.org.in' — space before @
    1967: 'llmrf@yahoo.com, landsteinercharities@gmail.com',                            # '/' separator → ','
    2194: 'labbalamurugan@gmail.com',                                                    # 'gmail,com' → 'gmail.com'
    2256: 'mnjiorcc@yahoo.com, dirmnjiorcc@yahoo.com',                                  # Merged at .com boundary
    2259: 'tscsap@gmail.com, tscsbb@gmail.com',                                         # Trailing '.' removed
}

fixed_count = 0
for sr_no, corrected_value in EMAIL_MANUAL_FIX.items():
    mask = df['Sr No'] == str(sr_no)
    if mask.sum() == 0:
        mask = df['Sr No'].astype(str) == str(sr_no)
    if mask.sum() > 0:
        original = df.loc[mask, 'Email'].values[0]
        new_val = corrected_value if corrected_value is not None else 'EMAIL_INVALID'
        df.loc[mask, 'Email'] = new_val
        fixed_count += 1
        print(f'  Sr {sr_no:4d}: {str(original)[:55]!r:60s} → {str(new_val)[:50]!r}')

print(f'\nTotal broken emails fixed/flagged: {fixed_count}')

  Sr   13: 'indian red cross society indian red corss society'          → 'EMAIL_INVALID'
  Sr   20: 'svimshosp'                                                  → 'EMAIL_INVALID'
  Sr   61: 'vijayasribloodbankvja @gmail.com'                           → 'vijayasribloodbankvja@gmail.com'
  Sr   76: 'knlgghbloodbank@gmail.comsupdtgghknl@gmail.com'             → 'knlgghbloodbank@gmail.com, supdtgghknl@gmail.com'
  Sr  152: 'dmoykggmail.com'                                            → 'EMAIL_INVALID'
  Sr  353: 'blood bankrjn@gmail.com'                                    → 'bloodbankrjn@gmail.com'
  Sr  465: 'adgbloodbank@charutarhealth.org / sadhanas@charutarheal'    → 'adgbloodbank@charutarhealth.org, sadhanas@charutar'
  Sr  467: 'glh_deesa@yahoo.com drjagdish2000@yahoo.co.in'              → 'glh_deesa@yahoo.com, drjagdish2000@yahoo.co.in'
  Sr  702: 'yes'                                                        → 'EMAIL_INVALID'
  Sr 1346: 'iimsk11@gmail.com, iimsr11@gmail.'            

---
## Fix 10 — Service Time: Normalise 24×7 Spelling Variants

~330 rows use non-standard spellings of `24x7`:  
`24X7`, `24 x 7`, `24*7`, `24 hrs`, `24 Hours`, `24BY 7`, `24H`, etc.  
All plain "round-the-clock" variants are standardised to `24x7`.  
Rows with real time ranges (e.g. `9 AM TO 4 PM...`) are left untouched.

In [12]:
# Variants that mean exactly "24x7" with no additional info
TWENTY_FOUR_VARIANTS = [
    '24X7', '24 x 7', '24 X 7', '24*7', '24BY 7',
    '24 Hours', '24 hours', '24 hours.', '24 HOURS',
    '24 hrs', '24 hrs.', '24 HRS', '24HRS', '24hrs', '24hrs.',
    '24 Hrs', '24 Hr.', '24 hr', '24 H', '24h',
    '24 hours 365 days',
]

before_count = (df['Service Time'] == '24x7').sum()

df['Service Time'] = df['Service Time'].replace(TWENTY_FOUR_VARIANTS, '24x7')

after_count = (df['Service Time'] == '24x7').sum()
print(f'Rows with "24x7": {before_count} → {after_count} (+{after_count - before_count} normalised)')

Rows with "24x7": 1807 → 2340 (+533 normalised)


---
## Fix 11 — Missing-Value Tokens: Standardise to Blank (NaN)

Fields inconsistently use `"NA"`, `"N/A"`, `"na"`, `"N-A"`, `"n/a"` to mean 'no value'.  
All such tokens are replaced with `NaN` (blank) for consistent null handling in any downstream pipeline.

In [13]:
NA_TOKENS = {'NA', 'N/A', 'N-A', 'na', 'n/a', 'n-a', 'N/a', 'Na'}

# Count before
before_total = 0
for col in df.columns:
    before_total += df[col].isin(NA_TOKENS).sum()

df = df.replace(list(NA_TOKENS), np.nan)

after_total = 0
for col in df.columns:
    after_total += df[col].isin(NA_TOKENS).sum()

print(f'NA-token cells replaced: {before_total} → {after_total} remaining')

NA-token cells replaced: 23 → 0 remaining


---
## Fix 12 — Dates: Normalise to `DD-MM-YYYY` + Extract Renewal Status

Both date columns use at least 7 different formats (`DD-MM-YYYY`, `DD.MM.YYYY`, `DD/MM/YYYY`,  
`DD.MM.YY`, year-only, `Mon-YY`, free text).  

`Date of Renewal` also contains status text (`Pending`, `Applied for renewal`, `Under Process`, etc.)  
which is not a date — this is extracted into a new column **`Renewal Status`**, and the date cell is left blank.

In [14]:
RENEWAL_STATUS_KEYWORDS = [
    'pending', 'applied', 'renewal', 'under process', 'under proceess',
    'process', 'a/f', 'awaited', 'await', 'not renewed', 'expired',
    'lapsed', 'cancelled', 'surrender', 'closed', 'fresh', 'due',
    'not applicable', 'exempted', 'not required', 'permanent',
]

DATE_FORMATS = [
    '%d-%m-%Y', '%d.%m.%Y', '%d/%m/%Y',
    '%d-%m-%y', '%d.%m.%y', '%d/%m/%y',
    '%Y',
]

def parse_date(v):
    """Try to parse a date string into DD-MM-YYYY; return (date_str, status_str)."""
    if pd.isna(v) or str(v).strip() == '':
        return (np.nan, np.nan)
    v_str = str(v).strip()

    # Check for status text
    v_lower = v_str.lower()
    for kw in RENEWAL_STATUS_KEYWORDS:
        if kw in v_lower:
            return (np.nan, v_str)

    # Try month-year pattern like 'Dec-11' or 'Dec-2011'
    m = re.match(r'^([A-Za-z]{3})[\-/](\d{2,4})$', v_str)
    if m:
        try:
            yr = m.group(2)
            yr_full = ('20' + yr) if len(yr) == 2 else yr
            dt = datetime.strptime(f'01-{m.group(1)}-{yr_full}', '%d-%b-%Y')
            return (dt.strftime('%d-%m-%Y'), np.nan)
        except ValueError:
            pass

    # Try known formats
    for fmt in DATE_FORMATS:
        try:
            dt = datetime.strptime(v_str, fmt)
            # Fix 2-digit year ambiguity: assume 00-30 → 2000s, 31-99 → 1900s
            if fmt in ('%d-%m-%y', '%d.%m.%y', '%d/%m/%y'):
                if dt.year < 1970:
                    dt = dt.replace(year=dt.year + 100)
            return (dt.strftime('%d-%m-%Y'), np.nan)
        except ValueError:
            continue

    # Year-only
    if re.match(r'^\d{4}$', v_str):
        return (f'01-01-{v_str}', np.nan)

    return (np.nan, v_str)  # Unparseable — move to status


# --- Date License Obtained ---
parsed_obtained = df['Date License Obtained'].apply(lambda v: parse_date(v)[0])
unparseable_obtained = df['Date License Obtained'][
    parsed_obtained.isna() & df['Date License Obtained'].notna()
].unique()

df['Date License Obtained'] = parsed_obtained

# --- Date of Renewal ---
renewal_results = df['Date of Renewal'].apply(parse_date)
df['Date of Renewal']  = renewal_results.apply(lambda x: x[0])
df['Renewal Status']   = renewal_results.apply(lambda x: x[1])

# Insert Renewal Status right after Date of Renewal
cols = df.columns.tolist()
dr_idx = cols.index('Date of Renewal')
cols.insert(dr_idx + 1, cols.pop(cols.index('Renewal Status')))
df = df[cols]

print('Date License Obtained — sample after fix:')
print(df['Date License Obtained'].dropna().head(5).tolist())

print('\nDate of Renewal — sample after fix:')
print(df['Date of Renewal'].dropna().head(5).tolist())

print('\nRenewal Status — value counts (top 10):')
print(df['Renewal Status'].value_counts().head(10))

if len(unparseable_obtained) > 0:
    print(f'\nUnparseable License Obtained values set to NaN: {unparseable_obtained.tolist()}')

Date License Obtained — sample after fix:
['14-06-1996', '14-06-1996', '08-09-2010', '22-11-2005', '28-01-1997']

Date of Renewal — sample after fix:
['01-01-2012', '31-12-2016', '10-06-2015', '21-11-2015', '01-01-2013']

Renewal Status — value counts (top 10):
Renewal Status
Pending                79
Applied for renewal    76
A/F renewal            44
Applied                37
Under Process          29
Applied for Renewal    14
31-11-2017             13
Not Available          11
31-12-2016 (R)         11
2012-2016               9
Name: count, dtype: int64

Unparseable License Obtained values set to NaN: ['06/10/1997-10/12/2013', '05/03/1997-2008', '11.06.2009 to 10.06.2014', 'License Cancelled- non functional', 'Not Available', '4.2015', 'Latest renwed License is awaited', '01/01/2012  to', 'Applied for renewal', 'applied for renewal', '-2011', '-2012', '-2013', '11-Dec', '01/01/2012 to', '06/02/2014 to', 'APPLIED FOR RENEWAL', '01-May', '1976 as BCSU-2003', '9/10/84-20/10/82', '27-06

---
## Fix 13 — Remove Self-Flagged `(Repeated)` / `(REPEATED)` Rows

6 rows have `(Repeated)` or `(REPEATED)` literally appended to the Blood Bank Name.  
These were identified as duplicates by the original data compiler but never removed.  
They are dropped from the dataset. The Sr No sequence is preserved as-is (not re-indexed).

In [15]:
repeated_mask = df['Blood Bank Name'].str.contains(
    r'\(Repeated\)|\(REPEATED\)', case=False, na=False, regex=True
)

print(f'Self-flagged repeated rows to remove ({repeated_mask.sum()}):')
print(df.loc[repeated_mask, ['Sr No', 'Blood Bank Name', 'City', 'State']].to_string(index=False))

df = df[~repeated_mask].reset_index(drop=True)

print(f'\nRows after removal: {len(df)}')

Self-flagged repeated rows to remove (6):
Sr No                                                             Blood Bank Name       City          State
   52                                                    Area Hospital (Repeated)     Tenali Andhra Pradesh
   58                                      Government General Hospital (Repeated) Vijayawada Andhra Pradesh
   67                                      Rotary Red Cross Blood Bank (Repeated) Vijayawada Andhra Pradesh
   91 M/s. Kanamarlapudi Koteswara Rao Indian Red Cross Society (IRCS) (REPEATED)     Kavali Andhra Pradesh
   95                                   District Headquarters Hospital (Repeated)    Nellore Andhra Pradesh
  132                      Indian Red Cross Society Blood Bank Tanuku  (Repeated)     Tanuku Andhra Pradesh

Rows after removal: 2817


---
## Fix 14 — Global Casing Uniformity

Logical casing rules applied per column:

| Column | Rule | Reason |
|--------|------|--------|
| Blood Bank Name | Title Case | Proper noun — names of institutions |
| State | Title Case | Proper noun |
| District | Title Case | Already done in Fix 7 — reaffirmed here |
| City | Title Case | Already done in Fix 5/6 — reaffirmed |
| Address | Title Case | Free-text address |
| Category | UPPER CASE | Short code-like field (`GOVT`, `PVT`, etc.) |
| Blood Component Available | UPPER CASE | Medical abbreviations (e.g. `WB`, `PC`) |
| Apheresis | UPPER CASE | YES / NO / UNKNOWN |
| Service Time | Preserved | Mix of codes (`24x7`) and free text — no change |
| Email / Website | Lower Case | Convention for email/URL |
| Nodal Officer | Title Case | Personal name |

In [16]:
# Title Case columns
TITLE_CASE_COLS = [
    'Blood Bank Name', 'State', 'District', 'City', 'Address',
    'Nodal Officer'
]
for col in TITLE_CASE_COLS:
    if col in df.columns:
        df[col] = df[col].str.strip().str.title()

# UPPER CASE columns
UPPER_COLS = ['Category', 'Blood Component Available', 'Apheresis']
for col in UPPER_COLS:
    if col in df.columns:
        df[col] = df[col].str.strip().str.upper()

# Lower case columns
LOWER_COLS = ['Email', 'Website', 'Email Nodal Officer']
for col in LOWER_COLS:
    if col in df.columns:
        # Only lowercase actual email/url values, not flags like EMAIL_INVALID
        mask = df[col].notna() & ~df[col].str.startswith('EMAIL_', na=False)
        df.loc[mask, col] = df.loc[mask, col].str.lower()

# Qualification Nodal Officer — Title Case
if 'Qualification Nodal Officer' in df.columns:
    df['Qualification Nodal Officer'] = df['Qualification Nodal Officer'].str.strip().str.title()

print('Global casing uniformity applied.')
print('\nSample — Blood Bank Name (Title Case):', df['Blood Bank Name'].dropna().iloc[:3].tolist())
print('Sample — Category (UPPER):', df['Category'].dropna().unique()[:5].tolist())
print('Sample — Email (lower):', df['Email'].dropna().iloc[:3].tolist())

Global casing uniformity applied.

Sample — Blood Bank Name (Title Case): ['G.B. Pant Hospital Blood Bank', 'I.N.H.S. Dhanvantri', 'Pillar Health Centre Blood Bank']
Sample — Category (UPPER): ['GOVERNMENT', 'CHARITY', 'PRIVATE']
Sample — Email (lower): ['bbgbpant@gmail.com', 'pillarbloodbank2016@gmail.com', 'bloodbankgghatp@gmail.com']


---
## Summary: Changes Made

In [17]:
print('=' * 65)
print('          BLOOD BANK DATA CLEANING — SUMMARY')
print('=' * 65)
print(f'  Final dataset shape : {df.shape[0]} rows x {df.shape[1]} columns')
print(f'  Rows removed        : 6  (self-flagged duplicates)')
print(f'  New column added    : Renewal Status')
print()
print('  Fix 1  Apheresis "2" → UNKNOWN')
print('  Fix 2  Corrupted phone commas stripped')
print('  Fix 3  Pincode spaces removed')
print('  Fix 4  Invalid pincodes → INVALID_PINCODE')
print('  Fix 5  City case variants → Title Case')
print('  Fix 6  City spelling variants → canonical names')
print('  Fix 7  District → Title Case')
print('  Fix 8  HTML entities (&#39;) unescaped')
print('  Fix 9  Broken emails manually corrected / flagged EMAIL_INVALID')
print('  Fix 10 Service Time 24×7 variants → 24x7')
print('  Fix 11 NA/N-A/N/A tokens → blank (NaN)')
print('  Fix 12 Dates normalised to DD-MM-YYYY; renewal status split out')
print('  Fix 13 6 repeated rows removed')
print('  Fix 14 Casing uniformity enforced across all text columns')
print('=' * 65)
print()
print('Remaining items for MANUAL review (not auto-fixed):')
print('  - 44 reused license numbers (needs licensing authority cross-check)')
print('  - 370+ missing geocoordinates (needs re-geocoding via API)')
print('  - 20 name+city duplicate rows (likely license renewal artefacts)')
print('  - 11 rows where Longitude = Latitude (copy-paste geocoding error)')
print('  - Date logic errors: 2 rows with year 1905, 1 with year 2105, 3 renewal-before-obtained')

          BLOOD BANK DATA CLEANING — SUMMARY
  Final dataset shape : 2817 rows x 28 columns
  Rows removed        : 6  (self-flagged duplicates)
  New column added    : Renewal Status

  Fix 1  Apheresis "2" → UNKNOWN
  Fix 2  Corrupted phone commas stripped
  Fix 3  Pincode spaces removed
  Fix 4  Invalid pincodes → INVALID_PINCODE
  Fix 5  City case variants → Title Case
  Fix 6  City spelling variants → canonical names
  Fix 7  District → Title Case
  Fix 8  HTML entities (&#39;) unescaped
  Fix 9  Broken emails manually corrected / flagged EMAIL_INVALID
  Fix 10 Service Time 24×7 variants → 24x7
  Fix 11 NA/N-A/N/A tokens → blank (NaN)
  Fix 12 Dates normalised to DD-MM-YYYY; renewal status split out
  Fix 13 6 repeated rows removed
  Fix 14 Casing uniformity enforced across all text columns

Remaining items for MANUAL review (not auto-fixed):
  - 44 reused license numbers (needs licensing authority cross-check)
  - 370+ missing geocoordinates (needs re-geocoding via API)
  - 20 na

---
## Export — Save Cleaned Dataset

In [19]:
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Cleaned dataset saved to: {OUTPUT_FILE}')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

# Quick final validation
print('\n--- Final Validation Checks ---')
print(f'Apheresis "2" remaining         : {(df["Apheresis"] == "2").sum()}')
space_pattern_count = df["Pincode"].str.contains(r"\s", na=False).sum()
print(f'Pincodes with spaces remaining  : {space_pattern_count}')
print(f'HTML entities remaining         : {df.apply(lambda c: c.str.contains("&#", na=False).sum() if c.dtype == object else 0).sum()}')
print(f'NA-token cells remaining        : {sum(df[c].isin({"NA","N/A","na","N-A"}).sum() for c in df.columns)}')
print(f'(Repeated) rows remaining       : {df["Blood Bank Name"].str.contains(r"Repeated", case=False, na=False).sum()}')

Cleaned dataset saved to: blood_bank_master_clean.csv
Shape: 2817 rows x 28 columns

--- Final Validation Checks ---
Apheresis "2" remaining         : 0
Pincodes with spaces remaining  : 0
HTML entities remaining         : 0
NA-token cells remaining        : 0
(Repeated) rows remaining       : 0
